In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 64
EPOCHS      = 5
LR_TRAIN    = 0.001
NUM_SEEDS   = 50
STEPS       = 30
LR_DX       = 0.05
LAMBDA      = 1.5
THRESHOLD   = 0.5
SAVE_DIR    = "outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Device       : {DEVICE}")
print(f"PyTorch      : {torch.__version__}")
print(f"Epochs       : {EPOCHS}")
print(f"Seeds        : {NUM_SEEDS}")
print(f"Steps/seed   : {STEPS}")

Device       : cuda
PyTorch      : 2.10.0+cu128
Epochs       : 5
Seeds        : 50
Steps/seed   : 30


In [3]:
class Model1(nn.Module):
    """Simple 2-block CNN — baseline architecture."""
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # (1,28,28)→(32,28,28)
            nn.ReLU(),
            nn.MaxPool2d(2)                              # →(32,14,14)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1), # →(64,14,14)
            nn.ReLU(),
            nn.MaxPool2d(2)                              # →(64,7,7)
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        return self.classifier(x.view(x.size(0), -1))


class Model2(nn.Module):
    """Deeper 3-block CNN with Dropout."""
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=5, padding=2),  # 5×5 filters
            nn.ReLU(), nn.MaxPool2d(2)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 7 * 7, 256), nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 10)
        )
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        return self.classifier(x.view(x.size(0), -1))


class Model3(nn.Module):
    """Wide CNN with Batch Normalization."""
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),  # more filters
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        return self.classifier(x.view(x.size(0), -1))

print("Model 1:", sum(p.numel() for p in Model1().parameters()), "parameters")
print("Model 2:", sum(p.numel() for p in Model2().parameters()), "parameters")
print("Model 3:", sum(p.numel() for p in Model3().parameters()), "parameters")

Model 1: 421642 parameters
Model 2: 829194 parameters
Model 3: 879114 parameters


In [4]:
class NeuronCoverageTracker:
    """
    Attaches hooks to all ReLU layers and tracks
    which neurons have fired at least once.
    """
    def __init__(self, model, threshold=0.5):
        self.model     = model
        self.threshold = threshold
        self.covered_neurons = {}
        self.total_neurons   = {}
        self._hook_handles   = []
        self._register_hooks()

    def _register_hooks(self):
        for name, module in self.model.named_modules():
            if isinstance(module, nn.ReLU):
                handle = module.register_forward_hook(self._make_hook(name))
                self._hook_handles.append(handle)
                self.covered_neurons[name] = set()

    def _make_hook(self, layer_name):
        def hook(module, input, output):
            with torch.no_grad():
                out = output[0]
                out_flat = out.mean(dim=[1,2]) if out.dim()==3 else out
                if layer_name not in self.total_neurons:
                    self.total_neurons[layer_name] = out_flat.numel()
                active = (out_flat > self.threshold).nonzero(as_tuple=True)[0]
                self.covered_neurons[layer_name].update(active.tolist())
        return hook

    def get_coverage(self):
        total   = sum(self.total_neurons.get(k,0) for k in self.covered_neurons)
        covered = sum(len(v) for v in self.covered_neurons.values())
        return covered / total if total > 0 else 0.0

    def get_coverage_percent(self):
        return self.get_coverage() * 100

    def get_uncovered_neurons(self):
        """Returns uncovered neuron indices per layer (used by optimizer)."""
        uncovered = {}
        for name, total in self.total_neurons.items():
            covered = self.covered_neurons[name]
            uncovered[name] = list(set(range(total)) - covered)
        return uncovered

    def reset(self):
        for k in self.covered_neurons:
            self.covered_neurons[k] = set()

    def remove_hooks(self):
        for h in self._hook_handles:
            h.remove()
        self._hook_handles = []

    def summary(self):
        print(f"{'Layer':<35} {'Covered':>8} {'Total':>8} {'Coverage':>10}")
        print("-" * 65)
        for name in self.covered_neurons:
            cov   = len(self.covered_neurons[name])
            total = self.total_neurons.get(name, 0)
            pct   = cov/total*100 if total>0 else 0
            print(f"{name:<35} {cov:>8} {total:>8} {pct:>9.1f}%")
        print("-" * 65)
        print(f"{'TOTAL':<35} {'':<8} {'':<8} {self.get_coverage_percent():>9.1f}%")

print("NeuronCoverageTracker defined ✓")

NeuronCoverageTracker defined ✓


In [5]:
class DeepXplore:
    def __init__(self, models, device, lambda_=1.0, threshold=0.5):
        self.models    = models
        self.device    = device
        self.lambda_   = lambda_
        self.threshold = threshold
        self._activations = {}
        for m in self.models:
            m.eval().to(device)

    def _register_hooks(self, model, model_idx):
        hooks = []
        for name, module in model.named_modules():
            if isinstance(module, nn.ReLU):
                key = f"m{model_idx}_{name}"
                def make_hook(k):
                    def hook(mod, inp, out): self._activations[k] = out
                    return hook
                hooks.append(module.register_forward_hook(make_hook(key)))
        return hooks

    def _coverage_loss(self, model_idx, uncovered_per_layer):
        loss = torch.tensor(0.0, device=self.device)
        for key, activation in self._activations.items():
            if not key.startswith(f"m{model_idx}_"): continue
            layer = key[len(f"m{model_idx}_"):]
            uncovered = uncovered_per_layer.get(layer, [])
            if not uncovered: continue
            act = activation.mean(dim=[2,3]) if activation.dim()==4 else activation
            idx = torch.tensor(uncovered, device=self.device)
            idx = idx[idx < act.shape[1]]
            if idx.numel() == 0: continue
            loss = loss + act[0, idx].mean()
        return loss


    def _differential_loss(self, outputs):
        probs = torch.stack([F.softmax(o, dim=1) for o in outputs])
        return probs.var(dim=0).sum()

    def _check_disagreement(self, outputs):
        preds = [o.argmax(dim=1).item() for o in outputs]
        return len(set(preds)) > 1, preds

    def generate(self, seed_input, coverage_trackers, steps=50, lr=0.01, perturbation="noise"):
        x = seed_input.clone().detach().to(self.device).requires_grad_(True)
        optimizer = torch.optim.Adam([x], lr=lr)
        best, found, best_preds = x.clone().detach(), False, []

        for step in range(steps):
            optimizer.zero_grad()
            self._activations.clear()

            all_hooks = []
            for i, m in enumerate(self.models):
                all_hooks.extend(self._register_hooks(m, i))

            outputs = [m(x) for m in self.models]

            for h in all_hooks: h.remove()

            # Coverage loss (sum over all models)
            cov_loss = sum(
                self._coverage_loss(i, t.get_uncovered_neurons())
                for i, t in enumerate(coverage_trackers)
            )

            # Differential loss
            diff_loss = self._differential_loss(outputs)

            # Joint loss (minimize = maximize both)
            loss = -(cov_loss + self.lambda_ * diff_loss)
            loss.backward()

            # Modify gradient shape based on perturbation type
            if perturbation == "brightness":
                with torch.no_grad():
                    x.grad = x.grad.mean() * torch.ones_like(x.grad)

            optimizer.step()
            with torch.no_grad(): x.clamp_(0.0, 1.0)

            # Check disagreement after this step
            with torch.no_grad():
                curr_out = [m(x) for m in self.models]
                disagrees, preds = self._check_disagreement(curr_out)
            if disagrees:
                best, found, best_preds = x.clone().detach(), True, preds
                break

        if not found:
            best = x.clone().detach()
            with torch.no_grad():
                _, best_preds = self._check_disagreement([m(x) for m in self.models])

        return best, found, best_preds

    # ── Full testing pipeline over many seeds ─────────────────────────
    def run_testing(self, data_loader, coverage_trackers, num_seeds=100, steps=50, lr=0.01):
        bugs, all_gen, seed_count = [], [], 0

        print("\n" + "="*60)
        print("  DeepXplore Testing Started")
        print("="*60)

        for batch_imgs, batch_labels in data_loader:
            if seed_count >= num_seeds: break
            for i in range(batch_imgs.size(0)):
                if seed_count >= num_seeds: break
                seed  = batch_imgs[i].unsqueeze(0).to(self.device)
                label = batch_labels[i].item()

                for perturb in ["noise", "brightness"]:
                    gen, is_bug, preds = self.generate(
                        seed, coverage_trackers, steps=steps, lr=lr, perturbation=perturb)
                    all_gen.append(gen)
                    if is_bug:
                        bugs.append({"seed_idx": seed_count, "true_label": label,
                                     "predictions": preds, "perturbation": perturb, "image": gen})

                seed_count += 1
                if seed_count % 10 == 0:
                    covs = [t.get_coverage_percent() for t in coverage_trackers]
                    print(f"  Seeds: {seed_count:3d} | Bugs: {len(bugs):3d} | "
                          f"Coverage M1={covs[0]:.1f}% M2={covs[1]:.1f}% M3={covs[2]:.1f}%")

        final_covs = [t.get_coverage_percent() for t in coverage_trackers]
        print("="*60)
        print(f"  Seeds tested : {seed_count}")
        print(f"  Bugs found   : {len(bugs)}")
        print(f"  Final Coverage → M1:{final_covs[0]:.1f}% M2:{final_covs[1]:.1f}% M3:{final_covs[2]:.1f}%")
        print("="*60)
        return {"bugs": bugs, "final_coverages": final_covs,
                "total_seeds": seed_count, "generated_inputs": all_gen}

print("DeepXplore engine defined ✓")

DeepXplore engine defined ✓


In [6]:

class SyntheticMNIST(Dataset):
    """28×28 grayscale, 10 classes — same structure as real MNIST."""
    def __init__(self, n_samples=6000, img_size=28, n_classes=10, seed=42):
        torch.manual_seed(seed); np.random.seed(seed)
        self.labels = torch.randint(0, n_classes, (n_samples,))
        self.images = self._generate(n_samples, img_size)

    def _generate(self, n, size):
        imgs = torch.zeros(n, 1, size, size)
        for i in range(n):
            c   = self.labels[i].item()
            img = torch.zeros(size, size)
            r0  = (c // 4) * 8 + 2
            c0  = (c %  4) * 6 + 2
            img[r0:min(r0+8,size), c0:min(c0+8,size)] = 0.7 + 0.3*torch.rand(
                min(8,size-r0), min(8,size-c0))
            for k in range(size):
                j = (k + c*3) % size
                img[k,j] = max(img[k,j].item(), 0.4)
            img = ((img + 0.05*torch.randn(size,size)).clamp(0,1) - 0.1307) / 0.3081
            imgs[i,0] = img
        return imgs

    def __len__(self):  return len(self.labels)
    def __getitem__(self, idx): return self.images[idx], self.labels[idx]

def get_dataloaders():
    try:
        tf = transforms.Compose([transforms.ToTensor(),
                                  transforms.Normalize((0.1307,),(0.3081,))])
        train_ds = datasets.MNIST("./data", train=True,  download=True, transform=tf)
        test_ds  = datasets.MNIST("./data", train=False, download=True, transform=tf)
        print("Using real MNIST ✓")
    except Exception:
        print("MNIST unavailable — using synthetic dataset.")
        train_ds = SyntheticMNIST(n_samples=6000)
        test_ds  = SyntheticMNIST(n_samples=1000)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    print(f"Train: {len(train_ds):,} samples  |  Test: {len(test_ds):,} samples")
    return train_loader, test_loader

train_loader, test_loader = get_dataloaders()

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 477kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.43MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.51MB/s]

Using real MNIST ✓
Train: 60,000 samples  |  Test: 10,000 samples


In [7]:
# ── Train all 3 models ─────────────────────────────────────────────────────
def train_model(model, loader, name):
    model.to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR_TRAIN)
    criterion = nn.CrossEntropyLoss()
    history   = []
    print(f"\nTraining {name}...")
    for epoch in range(1, EPOCHS+1):
        model.train()
        total_loss = 0
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward(); optimizer.step()
            total_loss += loss.item()
        avg = total_loss / len(loader)
        history.append(avg)
        print(f"  Epoch {epoch}/{EPOCHS}  Loss: {avg:.4f}")
    return history

def evaluate(model, loader, name):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            correct += (model(imgs).argmax(1) == labels).sum().item()
            total   += labels.size(0)
    acc = correct/total*100
    print(f"  {name} accuracy: {acc:.2f}%")
    return acc

# Build + train
model1, model2, model3 = Model1(), Model2(), Model3()
models      = [model1, model2, model3]
model_names = ["Model 1 (Simple CNN)", "Model 2 (Deep CNN)", "Model 3 (Wide+BN CNN)"]

histories   = [train_model(m, train_loader, n) for m, n in zip(models, model_names)]

print("\nEvaluation:")
accuracies  = [evaluate(m, test_loader, n) for m, n in zip(models, model_names)]


Training Model 1 (Simple CNN)...
  Epoch 1/5  Loss: 0.1261
  Epoch 2/5  Loss: 0.0398
  Epoch 3/5  Loss: 0.0270
  Epoch 4/5  Loss: 0.0191
  Epoch 5/5  Loss: 0.0145

Training Model 2 (Deep CNN)...
  Epoch 1/5  Loss: 0.1533
  Epoch 2/5  Loss: 0.0550
  Epoch 3/5  Loss: 0.0402
  Epoch 4/5  Loss: 0.0297
  Epoch 5/5  Loss: 0.0256

Training Model 3 (Wide+BN CNN)...
  Epoch 1/5  Loss: 0.1269
  Epoch 2/5  Loss: 0.0457
  Epoch 3/5  Loss: 0.0345
  Epoch 4/5  Loss: 0.0264
  Epoch 5/5  Loss: 0.0203

Evaluation:
  Model 1 (Simple CNN) accuracy: 99.22%
  Model 2 (Deep CNN) accuracy: 99.15%
  Model 3 (Wide+BN CNN) accuracy: 98.96%


In [8]:
# ── Plot training curves ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
for hist, name in zip(histories, ["Model 1", "Model 2", "Model 3"]):
    ax.plot(range(1, len(hist)+1), hist, marker='o', label=name)
ax.set_title("Training Loss per Epoch", fontsize=13)
ax.set_xlabel("Epoch"); ax.set_ylabel("Cross-Entropy Loss")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/training_curves.png", dpi=120)
plt.show()
print("Training complete ✓")

Training complete ✓


In [9]:
# ── Setup trackers ─────────────────────────────────────────────────────────
trackers = [NeuronCoverageTracker(m, threshold=THRESHOLD) for m in models]

print("Warming up coverage with test data...")
for model, tracker in zip(models, trackers):
    model.eval()
    with torch.no_grad():
        for imgs, _ in test_loader:
            _ = model(imgs.to(DEVICE))
            break

for tracker, name in zip(trackers, ["Model 1", "Model 2", "Model 3"]):
    print(f"  {name} warm-up coverage: {tracker.get_coverage_percent():.1f}%")

Warming up coverage with test data...
  Model 1 warm-up coverage: 19.2%
  Model 2 warm-up coverage: 15.8%
  Model 3 warm-up coverage: 6.6%


In [10]:
# ── Run DeepXplore ─────────────────────────────────────────────────────────
dx = DeepXplore(models, device=DEVICE, lambda_=LAMBDA, threshold=THRESHOLD)

results = dx.run_testing(
    test_loader,
    trackers,
    num_seeds = NUM_SEEDS,
    steps     = STEPS,
    lr        = LR_DX
)


  DeepXplore Testing Started
  Seeds:  10 | Bugs:  20 | Coverage M1=42.4% M2=45.9% M3=18.4%
  Seeds:  20 | Bugs:  40 | Coverage M1=45.5% M2=47.8% M3=19.4%
  Seeds:  30 | Bugs:  60 | Coverage M1=46.4% M2=47.8% M3=19.4%
  Seeds:  40 | Bugs:  80 | Coverage M1=46.4% M2=48.4% M3=19.4%
  Seeds:  50 | Bugs: 100 | Coverage M1=46.4% M2=48.6% M3=19.4%
  Seeds tested : 50
  Bugs found   : 100
  Final Coverage → M1:46.4% M2:48.6% M3:19.4%


In [11]:
# ── Per-layer coverage breakdown ───────────────────────────────────────────
for tracker, name in zip(trackers, ["Model 1", "Model 2", "Model 3"]):
    print(f"\n{name}")
    tracker.summary()


Model 1
Layer                                Covered    Total   Coverage
-----------------------------------------------------------------
block1.1                                   5       32      15.6%
block2.1                                   4       64       6.2%
classifier.1                              95      128      74.2%
-----------------------------------------------------------------
TOTAL                                                      46.4%

Model 2
Layer                                Covered    Total   Coverage
-----------------------------------------------------------------
block1.1                                   0       16       0.0%
block2.1                                   6       32      18.8%
block2.3                                   0       64       0.0%
classifier.1                             173      256      67.6%
-----------------------------------------------------------------
TOTAL                                                      48.6%

Mo

In [12]:
# ── Neuron Coverage Bar Chart ──────────────────────────────────────────────
coverages = [t.get_coverage_percent() for t in trackers]
colors    = ["#4C72B0", "#DD8452", "#55A868"]
labels    = ["Model 1", "Model 2", "Model 3"]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, coverages, color=colors, edgecolor="white", width=0.5)
for bar, val in zip(bars, coverages):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.5,
            f"{val:.1f}%", ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylim(0, 110)
ax.set_title("Neuron Coverage After DeepXplore Testing", fontsize=13)
ax.set_ylabel("Coverage (%)")
ax.axhline(100, color='red', linestyle='--', alpha=0.4, label='Max (100%)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/neuron_coverage.png", dpi=120)
plt.show()

In [13]:
# ── Bug-Triggering Input Examples ─────────────────────────────────────────
bugs    = results["bugs"]
mean, std = 0.1307, 0.3081   # MNIST normalization constants

if not bugs:
    print("No disagreements found — try increasing NUM_SEEDS or STEPS.")
else:
    n   = min(len(bugs), 6)
    fig = plt.figure(figsize=(14, 3*n))
    gs  = gridspec.GridSpec(n, 3, figure=fig, wspace=0.05, hspace=0.6)

    for row, bug in enumerate(bugs[:n]):
        img   = bug["image"].squeeze().cpu().numpy() * std + mean
        img   = np.clip(img, 0, 1)
        preds = bug["predictions"]
        mnames = ["Model 1", "Model 2", "Model 3"]

        for col in range(3):
            ax = fig.add_subplot(gs[row, col])
            ax.imshow(img, cmap='gray', vmin=0, vmax=1)
            color = "red" if len(set(preds))>1 else "green"
            ax.set_title(f"{mnames[col]}\nPredicts: {preds[col]}",
                         fontsize=9, color=color)
            ax.axis('off')

    fig.suptitle("Bug-Triggering Inputs Found by DeepXplore\n"
                 "(Red title = disagreement detected)", fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/bug_examples.png", dpi=120, bbox_inches='tight')
    plt.show()
    print(f"Total bugs found: {len(bugs)}")

/tmp/ipykernel_12012/173023554.py:28: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Total bugs found: 100


In [14]:
# ── Summary Dashboard ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Coverage
axes[0].bar(["M1","M2","M3"], results["final_coverages"], color=colors, edgecolor='white')
axes[0].set_title("Final Neuron Coverage (%)"); axes[0].set_ylim(0, 110)
for i,v in enumerate(results["final_coverages"]):
    axes[0].text(i, v+1, f"{v:.1f}%", ha='center', fontsize=10, fontweight='bold')

# Stats table
total = results["total_seeds"]; bugs_n = len(results["bugs"])
axes[1].axis('off')
table_data = [
    ["Seeds tested",    str(total)],
    ["Bugs found",      str(bugs_n)],
    ["Bug rate",        f"{bugs_n/max(total,1)*100:.1f}%"],
    ["Model 1 Acc",     f"{accuracies[0]:.1f}%"],
    ["Model 2 Acc",     f"{accuracies[1]:.1f}%"],
    ["Model 3 Acc",     f"{accuracies[2]:.1f}%"],
]
tbl = axes[1].table(cellText=table_data, colLabels=["Metric","Value"],
                    loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.2, 1.8)
axes[1].set_title("DeepXplore Results Summary")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/summary_dashboard.png", dpi=120)
plt.show()

In [15]:
# ── Bug Detail Table ───────────────────────────────────────────────────────
if bugs:
    print(f"{'#':>3}  {'True Label':>10}  {'M1':>4}  {'M2':>4}  {'M3':>4}  Perturbation")
    print("-" * 52)
    for i, bug in enumerate(bugs[:10]):
        p = bug['predictions']
        flag = "⚠" if len(set(p))>1 else " "
        print(f"{i+1:>3}  {bug['true_label']:>10}  {p[0]:>4}  {p[1]:>4}  {p[2]:>4}  "
              f"{bug['perturbation']}  {flag}")

for t in trackers: t.remove_hooks()
print("\nAll done! Outputs saved to ./outputs/")

  #  True Label    M1    M2    M3  Perturbation
----------------------------------------------------
  1           7     7     7     8  noise  ⚠
  2           7     7     7     8  brightness  ⚠
  3           2     2     2     8  noise  ⚠
  4           2     2     2     8  brightness  ⚠
  5           1     1     1     8  noise  ⚠
  6           1     1     1     8  brightness  ⚠
  7           0     0     0     8  noise  ⚠
  8           0     0     0     8  brightness  ⚠
  9           4     4     4     8  noise  ⚠
 10           4     4     4     8  brightness  ⚠

All done! Outputs saved to ./outputs/
